In [ ]:
%pip install uv

In [ ]:
!uv pip install --upgrade transformers datasets peft accelerate bitsandbytes qwen-vl-utils trackio
!uv pip install unsloth --torch-backend=auto
!uv pip install --upgrade ipywidgets widgetsnbextension jupyterlab_widgets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("naver-clova-ix/cord-v2")
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [ ]:
print(len(train_dataset))
print(len(eval_dataset))
print(len(test_dataset))

In [ ]:
print(train_dataset[10])

In [ ]:
system_prompt = "Extract all line items, quantities, and prices from this receipt as a JSON object"

def format_data(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text", "text": system_prompt}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["ground_truth"]}
                ]
            }
        ]
    }

In [ ]:
train_dataset = [format_data(sample) for sample in train_dataset]
eval_dataset = [format_data(sample) for sample in eval_dataset]
test_dataset = [format_data(sample) for sample in test_dataset]

In [ ]:
train_dataset[200]

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor

model_id = "Qwen/Qwen2-VL-7B-Instruct"

In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2VLProcessor.from_pretrained(model_id)

In [ ]:
from qwen_vl_utils import process_vision_info

def generate_text_from_sample(model, processor, sample, device, max_new_tokens=1024):
    # Extract only the user prompt turn (excluding target assistant response)
    user_messages = sample["messages"][:1]

    # Prepare text prompt with generation token
    text_input = processor.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Process vision inputs directly from the message structure
    image_inputs, video_inputs = process_vision_info(user_messages)

    # Move tensors to target device
    model_inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device)

    # Generate response
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Strip prompt token IDs from output
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode generated tokens
    output_text = processor.batch_decode(
        trimmed_generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return output_text[0]

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

In [ ]:
output = generate_text_from_sample(model, processor, train_dataset[0], device=device)
output